# Vine copula examples

R-vine fitting, comparison, sampling, prediction, conditional sampling, and RNG reproducibility.

In [1]:
import time
from pathlib import Path

import pandas as pd
import numpy as np
from matplotlib import pyplot as plt

from pyscarcopula._utils import pobs
from pyscarcopula import (
    GumbelCopula, FrankCopula, JoeCopula, ClaytonCopula,
    GaussianCopula, StudentCopula, CVineCopula, RVineCopula,
    PredictConfig,
)
from pyscarcopula.api import fit, sample, predict, smoothed_params
from pyscarcopula.stattests import gof_test

DATA_DIR = Path('data')
if not DATA_DIR.exists():
    DATA_DIR = Path('..') / 'data'


## Read dataset

In [2]:
crypto_prices = pd.read_csv(DATA_DIR / 'crypto_prices.csv', index_col=0, sep=';')

## 6. Multivariate R-vine copula

In [3]:
# truncated vines in this example

tickers_6d = ['BTC-USD', 'ETH-USD', 'BNB-USD', 'ADA-USD', 'XRP-USD', 'DOGE-USD']
returns_6d = np.log(crypto_prices[tickers_6d] / crypto_prices[tickers_6d].shift(1))[1:251].values
u_6d = pobs(returns_6d)

vine = RVineCopula()
vine.fit(u_6d, method='scar-tm-ou', truncation_level = 2)
vine.summary()

RVineCopula(d=6, T=250, criterion='aic')
  log_likelihood = 885.1891
  n_parameters   = 27
  AIC = -1716.3782   BIC = -1621.2987

Structure matrix (natural order, Czado 2019 Alg. 5.4; anti-diagonal = leaf peeled at each column):
[[3 5 5 5 5 5]
 [4 0 0 0 0 0]
 [5 4 1 1 0 0]
 [0 1 4 0 0 0]
 [1 3 0 0 0 0]
 [2 0 0 0 0 0]]

Edges (tree t, column col):
   t  col       pair           cond  family              rot  dyn_params                                       param     tau       logL
   0    0      (2,1)              -  GumbelCopula        180  theta= 50.848, mu=  2.650, nu= 13.637                     0.647    184.596
   0    1      (3,1)              -  GumbelCopula        180  theta= 12.932, mu=  1.954, nu=  3.459                     0.588    151.334
   0    2      (4,1)              -  GumbelCopula        180  theta= 61.802, mu=  2.521, nu=  9.179                     0.673    192.308
   0    3      (1,0)              -  GumbelCopula        180  theta= 18.506, mu=  2.464, nu=  5.144     

In [4]:
# GoF comparison: vine scar vs vine mle vs elliptical
vine_mle = RVineCopula()
vine_mle.fit(u_6d, method='mle', truncation_level = 2)

copula_s = StudentCopula()
copula_g = GaussianCopula()
copula_s.fit(u_6d)
copula_g.fit(u_6d)

gof_vine = gof_test(vine, u_6d, to_pobs=False)
gof_vine_mle = gof_test(vine_mle, u_6d, to_pobs=False)

gof_gauss = gof_test(copula_g, u_6d, to_pobs=False)
gof_stud = gof_test(copula_s, u_6d, to_pobs=False)

print(f"{'Model':<20} {'logL':>10} {'GoF p-value':>12}")
print('-' * 44)
print(f"{'R-vine SCAR-TM':<20} {vine.log_likelihood():>10.2f} {gof_vine.pvalue:>12.4f}")
print(f"{'R-vine MLE':<20} {vine_mle.log_likelihood():>10.2f} {gof_vine_mle.pvalue:>12.4f}")
print(f"{'Student-t':<20} {copula_s.log_likelihood(u_6d):>10.2f} {gof_stud.pvalue:>12.4f}")
print(f"{'Gaussian':<20} {copula_g.log_likelihood(u_6d):>10.2f} {gof_gauss.pvalue:>12.4f}")

Model                      logL  GoF p-value
--------------------------------------------
R-vine SCAR-TM           885.19       0.9839
R-vine MLE               836.96       0.0639
Student-t                764.42       0.0001
Gaussian                 761.00       0.0000


## 7. Vine sampling and prediction

In [5]:
# Vine model validation
v6 = vine.sample(2000)
gof_v6 = gof_test(vine, v6, to_pobs=False)
print(f'GoF on vine sample: p={gof_v6.pvalue:.4f}')

# Vine prediction
u_pred_6d = vine.predict(10000, u_train=u_6d, horizon='next')
print(f'Predicted shape: {u_pred_6d.shape}')

GoF on vine sample: p=0.6903
Predicted shape: (10000, 6)


### Conditional sampling for R-vine copulas

`RVineCopula.predict(..., given={var_index: u_value})` fixes variables in pseudo-observation space and samples the remaining coordinates conditionally.

The exact fast path supports `given` sets whose variables are at the end of the R-vine variable order, read from the anti-diagonal of the natural-order matrix. For some fitted structures, the same tree can be rebuilt into an equivalent natural-order matrix where the requested variables are last.

Arbitrary `given` patterns use the runtime-DAG initializer plus MCMC refinement (`conditional_method='dag_mcmc'`). Pass `return_diagnostics=True` to inspect which path was used, MCMC acceptance rates, and dynamic-conditioning updates.

Pass an explicit `np.random.Generator` through `rng` when you need reproducible conditional samples.

In [6]:
variable_order = [int(vine.matrix[vine.d - 1 - col, col]) for col in range(vine.d)]
print('R-vine variable order:', dict(enumerate(variable_order)))

# Case B: fixed variable is already last in the R-vine variable order.
given_order_tail = {variable_order[-1]: 0.4}

# Case C: not last in the current matrix, but placeable last after rebuild.
given_reordered = {variable_order[0]: 0.2}

# Arbitrary non-suffix pattern: handled by runtime-DAG + MCMC fallback.
given_arbitrary = {variable_order[0]: 0.2, variable_order[2]: 0.8}

print('Order-tail given:', given_order_tail)
print('Reordered-matrix given:', given_reordered)
print('Arbitrary DAG+MCMC given:', given_arbitrary)

R-vine variable order: {0: 2, 1: 3, 2: 4, 3: 1, 4: 0, 5: 5}
Order-tail given: {5: 0.4}
Reordered-matrix given: {2: 0.2}
Arbitrary DAG+MCMC given: {2: 0.2, 4: 0.8}


In [7]:
rng_exact = np.random.default_rng(2024)
rng_reordered = np.random.default_rng(2025)

t0 = time.perf_counter()
u_cond_order_tail = vine.predict(
    5000, u_train=u_6d, given=given_order_tail,
    horizon='current', rng=rng_exact,
)
t1 = time.perf_counter()

u_cond_reordered = vine.predict(
    5000, u_train=u_6d, given=given_reordered,
    horizon='next', rng=rng_reordered,
)
t2 = time.perf_counter()

print('Order-tail condition sample shape: ', u_cond_order_tail.shape)
print('Reordered-matrix condition sample shape:', u_cond_reordered.shape)
print(f'Order-tail condition time:  {t1 - t0:.3f}s')
print(f'Reordered-matrix condition time: {t2 - t1:.3f}s')

print('Fixed order-tail coordinate:')
for idx, val in given_order_tail.items():
    print(f'  {tickers_6d[idx]}:', np.unique(u_cond_order_tail[:, idx]))

print('Fixed reordered-matrix coordinate:')
for idx, val in given_reordered.items():
    print(f'  {tickers_6d[idx]}:', np.unique(u_cond_reordered[:, idx]))

print('Means of reordered-matrix conditional sample:')
print(pd.Series(u_cond_reordered.mean(axis=0), index=tickers_6d))


dag_config = PredictConfig(
    given=given_arbitrary,
    horizon='next',
    return_diagnostics=True,
    mcmc_steps=200,
    mcmc_burnin=80,
)
u_cond_arbitrary, dag_diag = vine.predict(
    2000, u_train=u_6d, predict_config=dag_config,
    rng=np.random.default_rng(2026),
)

print('Arbitrary DAG+MCMC sample shape:', u_cond_arbitrary.shape)
print('Conditional method:', dag_diag['conditional_method'])
print('MCMC acceptance min/mean/max:',
      round(dag_diag['mcmc']['acceptance_min'], 3),
      round(dag_diag['mcmc']['acceptance_mean'], 3),
      round(dag_diag['mcmc']['acceptance_max'], 3))

print('Fixed arbitrary coordinates:')
for idx, val in given_arbitrary.items():
    print(f'  {tickers_6d[idx]}:', np.unique(u_cond_arbitrary[:, idx]))

Order-tail condition sample shape:  (5000, 6)
Reordered-matrix condition sample shape: (5000, 6)
Order-tail condition time:  1.133s
Reordered-matrix condition time: 1.164s
Fixed order-tail coordinate:
  DOGE-USD: [0.4]
Fixed reordered-matrix coordinate:
  BNB-USD: [0.2]
Means of reordered-matrix conditional sample:
BTC-USD     0.346847
ETH-USD     0.327773
BNB-USD     0.200000
ADA-USD     0.368258
XRP-USD     0.354480
DOGE-USD    0.366289
dtype: float64
Arbitrary DAG+MCMC sample shape: (2000, 6)
Conditional method: dag_mcmc
MCMC acceptance min/mean/max: 0.414 0.453 0.519
Fixed arbitrary coordinates:
  BNB-USD: [0.2]
  XRP-USD: [0.8]


### Dynamic conditioning and RNG reproducibility

`dynamic_conditioning='given_only'` lets fixed `given` values update supported dynamic edge states before sampling the free coordinates. This is most useful when `given` represents observed current variables and the fitted vine has dynamic SCAR-TM/GAS edges.

Use a fresh generator with the same seed when you need identical Monte Carlo output across runs. Reusing the same generator object advances its stream.

In [8]:
# Use the two tail variables so the exact suffix sampler can condition dynamic edges.
given_tail_two = {
    variable_order[-2]: float(u_6d[-1, variable_order[-2]]),
    variable_order[-1]: float(u_6d[-1, variable_order[-1]]),
}

u_dyn, dyn_diag = vine.predict(
    3000,
    u_train=u_6d,
    given=given_tail_two,
    horizon='next',
    dynamic_conditioning='given_only',
    return_diagnostics=True,
    rng=np.random.default_rng(3030),
)

print('Dynamic-conditioning method:', dyn_diag['conditional_method'])
print('Updated dynamic edges:', dyn_diag['updated_edges'][:5])
print('Skipped dynamic edges:', dyn_diag['skipped_edges'][:5])
print('Means with dynamic conditioning:')
print(pd.Series(u_dyn.mean(axis=0), index=tickers_6d))

Dynamic-conditioning method: suffix
Updated dynamic edges: [{'key': (0, 4), 'tree': 0, 'col': 4, 'conditioned': (0, 5), 'conditioning': (), 'method': 'SCAR-TM-OU', 'family': 'GumbelCopula', 'status': 'updated', 'r_before_mean': 3.300110762523893, 'r_after_mean': 3.4242790004158077}]
Skipped dynamic edges: []
Means with dynamic conditioning:
BTC-USD     0.629482
ETH-USD     0.605862
BNB-USD     0.583094
ADA-USD     0.593389
XRP-USD     0.603118
DOGE-USD    0.633466
dtype: float64


In [9]:
seed = 4040
u_repro_a = vine.predict(
    1000, u_train=u_6d, given=given_order_tail,
    horizon='current', rng=np.random.default_rng(seed),
)
u_repro_b = vine.predict(
    1000, u_train=u_6d, given=given_order_tail,
    horizon='current', rng=np.random.default_rng(seed),
)
u_repro_c = vine.predict(
    1000, u_train=u_6d, given=given_order_tail,
    horizon='current', rng=np.random.default_rng(seed + 1),
)

print('Same seed -> identical:', np.allclose(u_repro_a, u_repro_b))
print('Different seed -> identical:', np.allclose(u_repro_a, u_repro_c))

Same seed -> identical: True
Different seed -> identical: False
